In [1]:
#kiểm tra có có tài khoản giống nhau giữa các ngân hàng không
import pandas as pd

In [2]:
account_node=pd.read_csv("dataset_high/HI-Small_accounts.csv")

In [3]:
account_node.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [4]:
df_clean = account_node[['Bank ID','Account Number']].drop_duplicates()

In [5]:
sum_unique=df_clean['Account Number'].nunique()

In [6]:
print(sum_unique)

518573


In [7]:
pairs=df_clean.groupby(['Account Number','Bank ID']).ngroups

In [8]:
print(pairs)

518581


In [ ]:
#kiểm tra dữ liệu split theo index
df_index_split = pd.read_csv("dataset_high/HI-Small_trans.csv")

In [13]:
order = df_index_split["Timestamp"].sort_values(kind="mergesort").index
df_index_split=df_index_split.iloc[order].reset_index(drop=True)
n=len(df_index_split)
t1=int(n*0.6)
t2=int(n*0.8)
ts1 = df_index_split["Timestamp"].iloc[t1]
ts2 = df_index_split["Timestamp"].iloc[t2]
# gán nhãn train/test/value cho tập dữ liệu mới và xuất file
df_index_split["split"]="test"
df_index_split.loc[df_index_split["Timestamp"]<ts2,"split"]="val"
df_index_split.loc[df_index_split["Timestamp"]<ts1,"split"]="train"

In [14]:
print(f"Tong so giao dich: {n:,}")
print(f"Diem cat raw: t1=index {t1} (ts1={ts1}), t2=index {t2} (ts2={ts2})")
print(f"Ranh gioi thuc te sau tie-breaking:")
print(f"  train max Timestamp = {df_index_split[df_index_split.split=='train']['Timestamp'].max()}")
print(f"  val   min Timestamp = {df_index_split[df_index_split.split=='val']['Timestamp'].min()}")
print(f"  val   max Timestamp = {df_index_split[df_index_split.split=='val']['Timestamp'].max()}")
print(f"  test  min Timestamp = {df_index_split[df_index_split.split=='test']['Timestamp'].min()}\n")

for s in ["train", "val", "test"]:
    sub = df_index_split[df_index_split.split == s]
    f = int((sub["Is Laundering"] == 1).sum())
    print(f"{s:5s} n={len(sub):>9,} ({len(sub)/n*100:5.2f}%)  "
          f"fraud={f:>5} rate={f/len(sub)*100:.4f}%  "
          f"ngay {sub['Timestamp'].min()[:10]}..{sub['Timestamp'].max()[:10]}")

Tong so giao dich: 5,078,345
Diem cat raw: t1=index 3047007 (ts1=2022/09/06 13:36), t2=index 4062676 (ts2=2022/09/08 16:12)
Ranh gioi thuc te sau tie-breaking:
  train max Timestamp = 2022/09/06 13:35
  val   min Timestamp = 2022/09/06 13:36
  val   max Timestamp = 2022/09/08 16:11
  test  min Timestamp = 2022/09/08 16:12

train n=3,046,861 (60.00%)  fraud= 2297 rate=0.0754%  ngay 2022/09/01..2022/09/06
val   n=1,015,602 (20.00%)  fraud= 1082 rate=0.1065%  ngay 2022/09/06..2022/09/08
test  n=1,015,882 (20.00%)  fraud= 1798 rate=0.1770%  ngay 2022/09/08..2022/09/18


In [16]:
split_index=pd.read_csv("dataset_high/HI-Small_Trans_split_index.csv")

In [17]:
split_index["Timestamp"] = pd.to_datetime(split_index["Timestamp"])
display(split_index[split_index["split"] == "val"][["Timestamp", "split"]].head())

,Timestamp,split
2766832,2022-09-06,val
2766833,2022-09-06,val
2766834,2022-09-06,val
2766835,2022-09-06,val
2766836,2022-09-06,val


,Timestamp,split
3046861,2022-09-06 13:36:00,val
3046862,2022-09-06 13:36:00,val
3046863,2022-09-06 13:36:00,val
3046864,2022-09-06 13:36:00,val
3046865,2022-09-06 13:36:00,val


In [18]:
print(split_index.iloc[3046850:3046860])

                  Timestamp  From Bank    Account  To Bank  Account.1  \
3046850 2022-09-06 13:35:00     226925  81232F7F0    48308  812AC0740   
3046851 2022-09-06 13:35:00     248857  8123B48C0      222  812D8CCD0   
3046852 2022-09-06 13:35:00         15  8000E2431       15  8000E2431   
3046853 2022-09-06 13:35:00        220  813634AE1      126  813764581   
3046854 2022-09-06 13:35:00     254595  813D936B1   254565  813F5F831   
3046855 2022-09-06 13:35:00        124  813D0C341      126  81409E921   
3046856 2022-09-06 13:35:00        225  813B7DE81       15  814135F31   
3046857 2022-09-06 13:35:00     152416  813CD5FB1   154148  814340251   
3046858 2022-09-06 13:35:00        220  813760121   152627  8144127A1   
3046859 2022-09-06 13:35:00        124  813DEFDC1   254595  8144306A1   

         Amount Received Receiving Currency   Amount Paid Payment Currency  \
3046850      7677.010000        Saudi Riyal   7677.010000      Saudi Riyal   
3046851     47177.020000        Saudi Ri

In [19]:
trans=pd.read_csv("dataset_high/HI-Small_Trans.csv")

In [20]:
trans["Timestamp"] = pd.to_datetime(trans["Timestamp"])

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
274314,2022-09-01 00:00:00,12,80BE909B0,12,80BE909B0,4464073.40,Mexican Peso,4464073.40,Mexican Peso,Reinvestment,0
29763,2022-09-01 00:00:00,2454,803BE7020,2454,803BE7020,5.59,US Dollar,5.59,US Dollar,Reinvestment,0
19444,2022-09-01 00:00:00,2947,802425F80,2947,802425F80,11730.44,US Dollar,11730.44,US Dollar,Reinvestment,0
181544,2022-09-01 00:00:00,17615,81037BF50,17615,81037BF50,16.72,Euro,16.72,Euro,Reinvestment,0
181540,2022-09-01 00:00:00,17769,80747D820,17615,81037BF50,2274855.54,Euro,2274855.54,Euro,Cheque,0
...,...,...,...,...,...,...,...,...,...,...,...
5033973,2022-09-18 09:55:00,29404,8041A3440,29404,8041A3440,2391.92,Saudi Riyal,637.66,US Dollar,ACH,0
5033974,2022-09-18 09:55:00,29404,8041A3440,118,811A79C40,2391.92,Saudi Riyal,2391.92,Saudi Riyal,ACH,1
4962229,2022-09-18 09:55:00,9371,8043A0FB0,24550,80FFF4150,9535.31,US Dollar,9535.31,US Dollar,ACH,1
4962232,2022-09-18 11:18:00,9371,8043A0FB0,13858,8095526B0,1785.27,Euro,1785.27,Euro,ACH,1


In [29]:
trans=trans.sort_values(by="Timestamp").sort_values(by="Timestamp").reset_index(drop=True)

In [30]:
trans.head(10)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:00,121,8123FB9B0,121,8123FB9B0,4.764000e+01,Saudi Riyal,4.764000e+01,Saudi Riyal,Reinvestment,0
1,2022/09/01 00:00,32317,800D4E490,12004,800D4E750,1.393905e+04,Euro,1.393905e+04,Euro,Wire,0
2,2022/09/01 00:00,1024,800C8D9D0,1024,800C8D9D0,1.037000e+01,Euro,1.037000e+01,Euro,Reinvestment,0
3,2022/09/01 00:00,21387,800C71170,21387,800C71170,6.667540e+03,Euro,6.667540e+03,Euro,Reinvestment,0
4,2022/09/01 00:00,254565,8140A8A71,254565,8140A8A71,1.058000e-03,Bitcoin,1.058000e-03,Bitcoin,Bitcoin,0
5,2022/09/01 00:00,1501,800C7C840,1502,801F5A2A0,2.578900e+02,Euro,2.578900e+02,Euro,Credit Card,0
6,2022/09/01 00:00,1669,800C72350,1669,800C72350,1.094000e+01,Euro,1.094000e+01,Euro,Reinvestment,0
7,2022/09/01 00:00,122055,80963BFC0,225734,809A49470,3.693434e+04,Canadian Dollar,3.693434e+04,Canadian Dollar,Credit Card,0
8,2022/09/01 00:00,12,800C83480,12,800C83480,1.186351e+06,Euro,1.186351e+06,Euro,Reinvestment,0
9,2022/09/01 00:00,21749,800C76180,21749,800C76180,1.274000e+01,Euro,1.274000e+01,Euro,Reinvestment,0


In [31]:
trans.drop_duplicates(subset=["Timestamp"], keep="first").head(10)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:00,121,8123FB9B0,121,8123FB9B0,47.64,Saudi Riyal,47.64,Saudi Riyal,Reinvestment,0
10977,2022/09/01 00:01,1674,8007B79E0,1674,8007B79E0,94324.82,US Dollar,94324.82,US Dollar,Reinvestment,0
21822,2022/09/01 00:02,11405,806E7FE70,11405,806E7FE70,2276.47,Euro,2667.53,US Dollar,ACH,0
32725,2022/09/01 00:03,147912,812FAE0F0,147912,812FAE0F0,104.24,Yuan,104.24,Yuan,Reinvestment,0
43547,2022/09/01 00:04,32311,800DB8520,12,800DB82D0,0.01,US Dollar,0.01,US Dollar,Cheque,0
54740,2022/09/01 00:05,144200,8112DA4C0,144200,8112DA4C0,8483.61,Shekel,8483.61,Shekel,Reinvestment,0
65693,2022/09/01 00:06,138105,8139654F0,138105,8139654F0,3.69,US Dollar,3.69,US Dollar,Reinvestment,0
76639,2022/09/01 00:07,70,1004289C0,45566,8114E6DC0,61561.59,Shekel,61561.59,Shekel,Credit Card,0
87537,2022/09/01 00:08,795,8017538C0,4766,80759C590,18299.43,US Dollar,18299.43,US Dollar,Credit Card,0
98395,2022/09/01 00:09,130449,80DC31190,130449,80DC31190,4051.55,US Dollar,4051.55,US Dollar,Reinvestment,0


In [32]:
Fraud=trans[trans["Is Laundering"]==1]

In [34]:
print(len(Fraud))

5177


In [35]:
Fraud.head(10)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
17111,2022/09/01 00:01,70,100428660,15980,80B39E7B0,792.92,US Dollar,792.92,US Dollar,Credit Card,1
33230,2022/09/01 00:03,70,100428660,113798,80DC756E0,13171425.53,US Dollar,13171425.53,US Dollar,Cheque,1
34778,2022/09/01 00:03,70,100428660,11474,805B716C0,29024.33,US Dollar,29024.33,US Dollar,Credit Card,1
43507,2022/09/01 00:03,1467,8013C4030,20,80BC62F10,58702.10,Yuan,58702.10,Yuan,ACH,1
45792,2022/09/01 00:04,119,811C597B0,48309,811C599A0,34254.65,Saudi Riyal,34254.65,Saudi Riyal,ACH,1
66095,2022/09/01 00:06,21174,800737690,12,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
67944,2022/09/01 00:06,70,100428858,8,8090A6540,66.06,Canadian Dollar,66.06,Canadian Dollar,Cash,1
108735,2022/09/01 00:09,70,100428738,10656,8044072A0,37080124.05,Yen,37080124.05,Yen,Cheque,1
141636,2022/09/01 00:12,70,1004286A8,24144,80D9B69A0,8873.07,Euro,8873.07,Euro,Cash,1
190674,2022/09/01 00:17,70,100428660,152980,8140702D0,6892.43,US Dollar,6892.43,US Dollar,Credit Card,1


In [39]:
# Lấy khối "Max 16-degree Fan-Out" trong patterns -> DataFrame
import io

# Dùng đúng bộ cột như HI-Small_Trans.csv (cột Account trùng -> pandas đổi thành Account.1)
cols = ["Timestamp","From Bank","Account","To Bank","Account.1","Amount Received",
        "Receiving Currency","Amount Paid","Payment Currency","Payment Format","Is Laundering"]

# Quét file, chỉ gom các dòng nằm giữa BEGIN ... và END của khối 16-degree fan-out
rows = []
inside = False
for line in open("dataset_high/HI-Small_Patterns.txt"):
    line = line.strip()
    if not line:
        continue
    if line.startswith("BEGIN LAUNDERING ATTEMPT - FAN-OUT") and "Max 16-degree" in line:
        inside = True
        continue
    if line.startswith("END LAUNDERING ATTEMPT"):
        if inside:
            break
        continue
    if inside:
        rows.append(line)

fan_out_16 = pd.read_csv(io.StringIO(chr(10).join(rows)), header=None, names=cols)
fan_out_16

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:06,21174,800737690,12,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
1,2022/09/01 04:33,21174,800737690,20,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1
2,2022/09/01 09:14,21174,800737690,20,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1
3,2022/09/01 09:56,21174,800737690,220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1
4,2022/09/01 11:28,21174,800737690,1244,80093C0D0,7254.53,US Dollar,7254.53,US Dollar,ACH,1
5,2022/09/01 13:13,21174,800737690,513,80078E200,6990.87,US Dollar,6990.87,US Dollar,ACH,1
6,2022/09/01 14:11,21174,800737690,20,80066B990,12536.92,Euro,12536.92,Euro,ACH,1
7,2022/09/02 15:40,21174,800737690,410,8002CC310,3511.82,Euro,3511.82,Euro,ACH,1
8,2022/09/02 21:23,21174,800737690,1292,8004030A0,16135.09,US Dollar,16135.09,US Dollar,ACH,1
9,2022/09/02 23:10,21174,800737690,1601,800578800,12183.28,US Dollar,12183.28,US Dollar,ACH,1


In [ ]:
import pandas as pd, numpy as np
df=pd.read_csv("dataset_high/HI-Small_Trans.csv",dtype={"From Bank":str, "Account":str, "To Bank": str, "Account.1":str})
df["src"]=df["From Bank"] + " | " + df["Account"]#cột tài khoản, node
df["dest"]=df["To Bank"] +" | " +df["Account.1"]
df["!currency"] = df["Receiving Currency"] != df["Payment Currency"]# tỷ lệ trộn trên tất cả các giao dịch
df["Is_Mule"] = df["Is Laundering"]==1
laund=df[df["Is_Mule"]]
set_mule=set(laund["src"]) | set(laund["dest"])
g=df.groupby("src")
receive=df.groupby("dest")["Amount Received"].sum().rename("tong_nhan")
df=g.agg(so_giao_dich_gui=("Amount Paid","size"),
         tong_chi=("Amount Paid","sum"),
         ti_le_tron_tien=("!currency","mean"),
         so_nguoi_nhan=("dest","nunique")
        )
df=df.join(receive)
df["tong_nhan"]=df["tong_nhan"].fillna(0)
df["net_flow"]=df["tong_nhan"]-df["tong_chi"]
df["Is_mule"]=df.index.isin(set_mule)
df.head()

,so_giao_dich_gui,tong_chi,ti_le_tron_tien,so_nguoi_nhan,tong_nhan,net_flow,Is_mule
src,,,,,,,
001 | 800042CB0,134,1.275401e+06,0.000000,13,40018.84,-1.235383e+06,False
001 | 800054E60,222,2.781327e+07,0.009009,13,975938.43,-2.683734e+07,False
001 | 8000555D0,128,1.596319e+08,0.000000,9,3158455.22,-1.564735e+08,False
001 | 800056160,89,2.106860e+05,0.000000,11,6227.20,-2.044588e+05,True
001 | 800056370,131,5.500107e+07,0.000000,9,81579.64,-5.491949e+07,False


In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
df=pd.read_csv("dataset_high/HI-Small_Trans.csv",dtype={"From Bank":str, "Account":str, "To Bank": str, "Account.1":str})
df["src"]=df["From Bank"] + " | " + df["Account"]#cột tài khoản, node
df["dest"]=df["To Bank"] +" | " +df["Account.1"]
df["!currency"] = df["Receiving Currency"] != df["Payment Currency"]# tỷ lệ trộn trên tất cả các giao dịch
df["Is_Mule"] = df["Is Laundering"]==1
laund=df[df["Is_Mule"]]
set_mule=set(laund["src"]) | set(laund["dest"])
g=df.groupby("src")
receive=df.groupby("dest")["Amount Received"].sum().rename("tong_nhan")
df=g.agg(so_giao_dich_gui=("Amount Paid","size"),
         tong_chi=("Amount Paid","sum"),
         ti_le_tron_tien=("!currency","mean"),
         so_nguoi_gui=("src","nunique"),
         so_nguoi_nhan=("dest","nunique"),
         mean_nhan=("tong_nhan","mean"),
         mean_chi=("tong_chi","mean"),
         Bank_tile=("From Bank","nunique")
        )
df=df.join(receive)
df["tong_nhan"]=df["tong_nhan"].fillna(0)
df["net_flow"]=df["tong_nhan"]-df["tong_chi"]
df["Is_mule"]=df.index.isin(set_mule)
df.head()

In [ ]:
# Khái quát trên CẢ nhóm "trả cho nhiều người" (out-degree >= 10) ===
nhom = df[df["so_nguoi_nhan"] >= 10]
print("Số tài khoản trả cho >=10 người:", len(nhom),
      "| trong đó mule:", int(nhom["la_mule"].sum()))
nhom.groupby("la_mule")[cols].median()   # so trung vị: bình thường vs mule

Số tài khoản trả cho >=10 người: 6078 | trong đó mule: 202


,so_nguoi_nhan,so_giao_dich_gui,tong_chi,tong_nhan,net_flow,ti_le_tron_tien
la_mule,,,,,,
False,11.0,53.0,817053.805,1296618.035,147768.635,0.000000
True,13.0,118.0,4646379.385,364660.935,-2977309.810,0.007576
